# GAN Workflow Example

This example shows how GANs can be run on images for the wildfire dataset by using the DSI framework.

## Downloading wildfire data
Follow steps in the ReadMe

## Importing and initializing GAN

In [ ]:
from gan import GAN

dataset_name = 'wildfire'
image_column = "FILE"

gWorkflow = GAN(dataset_name, image_column)

## Training data

In [ ]:
path_to_model = 'untrained_models'
epochs = 2
batch_size = 256
noise = 100
image_width = 64
image_height = 64

gWorkflow.load_model(path_to_model, gWorkflow.ModelType.generator)
gWorkflow.load_model(path_to_model, gWorkflow.ModelType.discriminator)

images = gWorkflow.shape_images(gWorkflow.imageArray, image_width, image_height)
images = gWorkflow.format_images(images)

gWorkflow.train(images, epochs, batch_size, noise)
gWorkflow.save_models() # saves models to wildfire/trained_models/

gWorkflow.plot_training_metrics("metrics.png")

## Predicting data

In [ ]:
path_to_model = 'pretrained_models/' # or 'trained_models/' if you have trained with data above
num_images = 16
noise_dim = 100
prediction_path = 'prediction.png'

gWorkflow.load_model(path_to_model, gWorkflow.ModelType.generator)
generatedImages = gWorkflow.predict(num_images, noise_dim, prediction_path)

## Evaluating data

In [ ]:
path_to_model = 'pretrained_models/' # or 'trained_models/' if you have trained with data above
image_width = 64
image_height = 64
num_images = 25
noise_dim = 100
num_samples = 4

gWorkflow.load_model(path_to_model, gWorkflow.ModelType.generator)
images = gWorkflow.shape_images(gWorkflow.imageArray, image_width, image_height)
images = gWorkflow.format_images(images)

generatedImages = gWorkflow.predict(num_images, noise_dim)
generatedImages = generatedImages.numpy()
IS = gWorkflow.calculate_inception_score(generatedImages)
print("Inception Score: ", IS)

FID, MSE = [], []
for i in range(num_images):
    MSE.append(gWorkflow.calculate_mse(images[i], generatedImages[i]))
    print(f"Mean Squared Error (MSE): {MSE[-1]}")
    FID.append(gWorkflow.calculate_fid(images[i], generatedImages[i]))
    print('Frechet Inception Distance: %.3f' % FID[-1])

metrics = [FID, MSE]
titles = ['FID', 'MSE']

gWorkflow.generate_scatterplot(metrics, titles, num_images, 'metrics_scatter_plots.png')

gWorkflow.generate_heatmaps(images, generatedImages, num_samples, 'generated_heatmaps.png')